# RavenX Chaos Agent 27B — Colab GPU autonome

Ce notebook installe **llama-cpp-python CUDA**, charge une seule instance de **RavenX Chaos Agent Qwen3.8 27B Q4_K_M** avec `Llama.from_pretrained`, exécute une inférence réelle, publie une API OpenAI compatible, ouvre un tunnel Cloudflare et l’enregistre sur le VPS.

## Secrets Colab requis

Dans le panneau **Secrets** de Colab, créer uniquement :

- `hex` : secret HMAC partagé avec le VPS ; il sert aussi de clé API Bearer.
- `vps` : `IP:PORT`, par exemple `203.0.113.10:8765`, ou une URL HTTPS complète.

Le dataset `r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation` est un corpus SFT optionnel, pas un modèle chargeable par llama.cpp. Il n’est donc pas téléchargé pour l’inférence.

Choisir un runtime GPU puis exécuter **Runtime → Run all**. La dernière cellule reste active pour maintenir le service. Le client recommandé est **OpenCode TUI/CLI**.


In [ ]:
# 1. Configuration immuable et secrets
import os, json, time, hmac, hashlib, subprocess, sys, shutil, re, gc
from pathlib import Path

MODEL_REPO = "deadbydawn101/RavenXAiLabs-Chaos-Agent-Qwen3.8-27B-Frontier-Intelligence-Injected-OBLITERATED-GGUF"
MODEL_FILE = "RavenX-Chaos-Agent-Q4_K_M.gguf"
MODEL_ALIAS = "ravenx-chaos-agent-qwen3.8-27b"
DISTILLATION_DATASET = "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation"
EXPECTED_SIZE = 16_547_399_968
ROOT = Path("/content/ravenx-colab")
MODEL_DIR = ROOT / "models"
LOG_DIR = ROOT / "logs"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata
    HB_SECRET = userdata.get("hex").strip()
    VPS = userdata.get("vps").strip()
except Exception as exc:
    raise RuntimeError("Secrets Colab requis: hex et vps") from exc
if len(HB_SECRET) < 16:
    raise ValueError("Le secret hex doit contenir au moins 16 caractères")
if not VPS:
    raise ValueError("Le secret vps est vide")
API_KEY = HB_SECRET
print(json.dumps({"model": MODEL_ALIAS, "repo": MODEL_REPO, "file": MODEL_FILE, "dataset_optional": DISTILLATION_DATASET, "vps_configured": True}, indent=2))


In [ ]:
# 2. Diagnostic matériel et paramètres adaptatifs T4/L4/A100/H100
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "psutil>=5.9"], check=True)
import psutil

def capture(args, timeout=60):
    return subprocess.run(args, check=False, capture_output=True, text=True, timeout=timeout)

gpu = capture(["nvidia-smi", "--query-gpu=name,memory.total,memory.free,compute_cap", "--format=csv,noheader,nounits"])
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("GPU NVIDIA requis. Sélectionner un runtime GPU Colab.")
parts = [x.strip() for x in gpu.stdout.splitlines()[0].split(",")]
GPU_NAME, VRAM_TOTAL_MIB, VRAM_FREE_MIB, COMPUTE_CAP = parts[0], int(parts[1]), int(parts[2]), parts[3]
VRAM_GIB = VRAM_TOTAL_MIB / 1024
RAM_GIB = psutil.virtual_memory().total / 1024**3
DISK_FREE_GIB = shutil.disk_usage("/content").free / 1024**3
if DISK_FREE_GIB < 22:
    raise RuntimeError(f"Espace disque insuffisant: {DISK_FREE_GIB:.1f} GiB; 22 GiB requis")

# Q4_K_M pèse 15.41 GiB. Les essais décroissants évitent un échec définitif OOM.
if VRAM_GIB < 18:  # T4 16 GiB
    CONTEXT, GPU_LAYER_CANDIDATES = 4096, [54, 48, 40, 32, 24]
elif VRAM_GIB < 35:  # L4 24 GiB
    CONTEXT, GPU_LAYER_CANDIDATES = 8192, [-1, 60, 56, 48]
else:  # A100/H100 40–80 GiB
    CONTEXT, GPU_LAYER_CANDIDATES = 16384, [-1, 64, 60]
THREADS = max(2, min(os.cpu_count() or 2, 8))
N_BATCH = min(512, CONTEXT)
print(json.dumps({"gpu": GPU_NAME, "vram_gib": round(VRAM_GIB, 2), "ram_gib": round(RAM_GIB, 2), "disk_free_gib": round(DISK_FREE_GIB, 2), "context": CONTEXT, "gpu_layer_candidates": GPU_LAYER_CANDIDATES}, indent=2))


In [ ]:
# 3. Installation CUDA reproductible de llama-cpp-python et de l'API
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "cmake", "ninja", "huggingface_hub>=0.34", "fastapi>=0.115", "uvicorn>=0.34", "requests>=2.32"], check=True)

# Les wheels CUDA évitent une longue compilation. Le fallback compile llama.cpp avec GGML_CUDA.
wheel = subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-cache-dir",
    "llama-cpp-python>=0.3.16", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124",
], check=False)
if wheel.returncode != 0:
    build_env = os.environ.copy()
    build_env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    build_env["FORCE_CMAKE"] = "1"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--force-reinstall", "--no-cache-dir", "llama-cpp-python>=0.3.16"], check=True, env=build_env)

from llama_cpp import Llama, llama_cpp
if llama_cpp.llama_supports_gpu_offload() is not True:
    raise RuntimeError("llama-cpp-python a été installé sans support GPU CUDA")
print("llama-cpp-python CUDA prêt")


In [ ]:
# 4. Chargement unique du GGUF exact avec Llama.from_pretrained
from huggingface_hub import list_repo_files
files = set(list_repo_files(MODEL_REPO, repo_type="model"))
if MODEL_FILE not in files:
    raise FileNotFoundError(f"{MODEL_FILE} absent du dépôt {MODEL_REPO}")

llm = None
SELECTED_GPU_LAYERS = None
load_errors = []
for ngl in GPU_LAYER_CANDIDATES:
    try:
        print("Chargement avec n_gpu_layers=", ngl)
        llm = Llama.from_pretrained(
            repo_id=MODEL_REPO,
            filename=MODEL_FILE,
            cache_dir=str(MODEL_DIR),
            n_ctx=CONTEXT,
            n_batch=N_BATCH,
            n_threads=THREADS,
            n_threads_batch=THREADS,
            n_gpu_layers=ngl,
            offload_kqv=True,
            flash_attn=True,
            use_mmap=True,
            chat_format=None,
            verbose=True,
        )
        SELECTED_GPU_LAYERS = ngl
        break
    except Exception as exc:
        load_errors.append(f"n_gpu_layers={ngl}: {type(exc).__name__}: {exc}")
        if llm is not None:
            llm.close()
        llm = None
        gc.collect()
if llm is None:
    raise RuntimeError("Impossible de charger RavenX:\n" + "\n".join(load_errors))

MODEL_PATH = Path(llm.model_path)
size = MODEL_PATH.stat().st_size
if size != EXPECTED_SIZE:
    llm.close()
    raise RuntimeError(f"GGUF incomplet: {size} octets, attendu {EXPECTED_SIZE}")
print(f"GGUF chargé une seule fois: {MODEL_PATH} ({size/1024**3:.2f} GiB), GPU layers={SELECTED_GPU_LAYERS}")


In [ ]:
# 5. Inférence directe réelle sur l'instance chargée
DIRECT_MESSAGES = [
    {"role": "system", "content": "Tu es un assistant de programmation précis. Réponds sans raisonnement caché."},
    {"role": "user", "content": "Réponds uniquement par OK."},
]
t0 = time.time()
direct_result = llm.create_chat_completion(
    messages=DIRECT_MESSAGES,
    temperature=0.0,
    repeat_penalty=1.15,
    max_tokens=32,
)
elapsed = time.time() - t0
direct_text = direct_result["choices"][0]["message"]["content"].strip()
completion_tokens = direct_result.get("usage", {}).get("completion_tokens", 0)
print(json.dumps({"direct_inference": direct_text, "elapsed_s": round(elapsed, 2), "completion_tokens": completion_tokens, "tokens_per_second": round(completion_tokens / elapsed, 2) if completion_tokens else None}, indent=2, ensure_ascii=False))


In [ ]:
# 6. API OpenAI compatible sur la même instance + tests models/chat/stream
import threading
import uuid
import requests
import uvicorn
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse

LLAMA_PORT = 8000
GENERATION_LOCK = threading.Lock()
app = FastAPI(title="RavenX Colab OpenAI API", version="1.0")
TOOL_BLOCK_RE = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.DOTALL)
FUNCTION_RE = re.compile(r"<function=([^>\s]+)>\s*(.*?)\s*</function>", re.DOTALL)
PARAMETER_RE = re.compile(r"<parameter=([^>\s]+)>\s*(.*?)\s*</parameter>", re.DOTALL)

def check_bearer(request):
    supplied = request.headers.get("authorization", "")
    expected = f"Bearer {API_KEY}"
    if not hmac.compare_digest(supplied, expected):
        raise HTTPException(status_code=401, detail="Invalid API key")

def completion_kwargs(body):
    if body.get("model") not in (None, MODEL_ALIAS):
        raise HTTPException(status_code=404, detail="Model not found")
    if not isinstance(body.get("messages"), list) or not body["messages"]:
        raise HTTPException(status_code=422, detail="messages must be a non-empty list")
    allowed = ("messages", "temperature", "top_p", "top_k", "min_p", "max_tokens", "stop", "stream", "frequency_penalty", "presence_penalty", "repeat_penalty", "seed", "tools", "tool_choice", "response_format")
    kwargs = {key: body[key] for key in allowed if key in body}
    if "max_completion_tokens" in body and "max_tokens" not in kwargs:
        kwargs["max_tokens"] = body["max_completion_tokens"]
    kwargs.setdefault("temperature", 0.0)
    kwargs.setdefault("repeat_penalty", 1.15)
    kwargs.setdefault("max_tokens", 2048)
    return kwargs

def decode_argument(value):
    value = value.strip()
    try:
        return json.loads(value)
    except (TypeError, ValueError):
        return value

def extract_xml_tool_calls(content):
    if not isinstance(content, str):
        return content, []
    calls = []
    for block in TOOL_BLOCK_RE.finditer(content):
        inner = block.group(1).strip()
        try:
            payload = json.loads(inner)
        except (TypeError, ValueError):
            payload = None
        if isinstance(payload, dict) and payload.get("name"):
            arguments = payload.get("arguments", {})
            if not isinstance(arguments, dict):
                arguments = {"value": arguments}
            name = payload["name"]
        else:
            function = FUNCTION_RE.search(inner)
            if not function:
                continue
            name = function.group(1)
            arguments = {
                match.group(1): decode_argument(match.group(2))
                for match in PARAMETER_RE.finditer(function.group(2))
            }
        calls.append({
            "id": "call_" + uuid.uuid4().hex[:24],
            "type": "function",
            "function": {"name": name, "arguments": json.dumps(arguments, ensure_ascii=False)},
        })
    clean = TOOL_BLOCK_RE.sub("", content).strip()
    return clean or None, calls

def normalize_completion(result):
    result["model"] = MODEL_ALIAS
    choice = result["choices"][0]
    message = choice["message"]
    if not message.get("tool_calls"):
        content, calls = extract_xml_tool_calls(message.get("content"))
        if calls:
            message["content"] = content
            message["tool_calls"] = calls
            choice["finish_reason"] = "tool_calls"
    return result

def sse_frame(payload):
    return "data: " + json.dumps(payload, ensure_ascii=False) + chr(10) * 2

def buffered_tool_stream(result):
    choice = result["choices"][0]
    message = choice["message"]
    delta = {"role": "assistant"}
    if message.get("content"):
        delta["content"] = message["content"]
    if message.get("tool_calls"):
        delta["tool_calls"] = [dict(call, index=index) for index, call in enumerate(message["tool_calls"])]
    chunk = {
        "id": result.get("id", "chatcmpl-" + uuid.uuid4().hex),
        "object": "chat.completion.chunk",
        "created": result.get("created", int(time.time())),
        "model": MODEL_ALIAS,
        "choices": [{"index": 0, "delta": delta, "finish_reason": None}],
    }
    yield sse_frame(chunk)
    chunk["choices"] = [{"index": 0, "delta": {}, "finish_reason": choice.get("finish_reason", "stop")}]
    yield sse_frame(chunk)
    yield "data: [DONE]" + chr(10) * 2

@app.get("/health")
def health(request: Request):
    check_bearer(request)
    return {"status": "ok", "model": MODEL_ALIAS}

@app.get("/v1/models")
def models(request: Request):
    check_bearer(request)
    return {"object": "list", "data": [{"id": MODEL_ALIAS, "object": "model", "created": int(time.time()), "owned_by": "colab-ravenx"}]}

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    check_bearer(request)
    try:
        body = await request.json()
        kwargs = completion_kwargs(body)
    except HTTPException:
        raise
    except Exception as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc

    # Avec des outils, bufferiser la réponse permet de convertir le XML Qwen en tool_calls OpenAI.
    if kwargs.get("stream") and kwargs.get("tools"):
        kwargs["stream"] = False
        with GENERATION_LOCK:
            try:
                result = normalize_completion(llm.create_chat_completion(**kwargs))
            except Exception as exc:
                raise HTTPException(status_code=500, detail=str(exc)) from exc
        return StreamingResponse(buffered_tool_stream(result), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

    if kwargs.get("stream"):
        def event_stream():
            with GENERATION_LOCK:
                try:
                    for chunk in llm.create_chat_completion(**kwargs):
                        chunk["model"] = MODEL_ALIAS
                        yield sse_frame(chunk)
                except Exception as exc:
                    yield sse_frame({"error": {"message": str(exc), "type": "server_error"}})
                yield "data: [DONE]" + chr(10) * 2
        return StreamingResponse(event_stream(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

    with GENERATION_LOCK:
        try:
            return JSONResponse(normalize_completion(llm.create_chat_completion(**kwargs)))
        except Exception as exc:
            raise HTTPException(status_code=500, detail=str(exc)) from exc

if globals().get("API_SERVER"):
    API_SERVER.should_exit = True
    time.sleep(1)
API_SERVER = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=LLAMA_PORT, log_level="info"))
SERVER_THREAD = threading.Thread(target=API_SERVER.run, daemon=True)
SERVER_THREAD.start()

AUTH_HEADERS = {"Authorization": f"Bearer {API_KEY}"}
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{LLAMA_PORT}/health", headers=AUTH_HEADERS, timeout=3).status_code == 200:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("L'API locale ne répond pas")

models_response = requests.get(f"http://127.0.0.1:{LLAMA_PORT}/v1/models", headers=AUTH_HEADERS, timeout=30)
models_response.raise_for_status()
chat_payload = {"model": MODEL_ALIAS, "messages": [{"role": "user", "content": "Réponds uniquement: API_OK"}], "temperature": 0, "max_tokens": 32}
chat_response = requests.post(f"http://127.0.0.1:{LLAMA_PORT}/v1/chat/completions", headers=AUTH_HEADERS, json=chat_payload, timeout=300)
chat_response.raise_for_status()
stream_payload = {**chat_payload, "stream": True, "max_tokens": 8}
with requests.post(f"http://127.0.0.1:{LLAMA_PORT}/v1/chat/completions", headers=AUTH_HEADERS, json=stream_payload, stream=True, timeout=300) as stream_response:
    stream_response.raise_for_status()
    stream_ok = any(line == b"data: [DONE]" for line in stream_response.iter_lines())
if not stream_ok:
    raise RuntimeError("Le flux SSE OpenAI ne s'est pas terminé correctement")
print(json.dumps({"local_api": f"http://127.0.0.1:{LLAMA_PORT}/v1", "models": [m["id"] for m in models_response.json()["data"]], "chat": chat_response.json()["choices"][0]["message"]["content"].strip(), "stream": "OK", "tool_call_adapter": "OpenAI"}, indent=2, ensure_ascii=False))


In [ ]:
# 7. Tunnel Cloudflare avec contrôle de disponibilité
CF_BIN = ROOT / "cloudflared"
if not CF_BIN.exists():
    subprocess.run(["curl", "-fL", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-o", str(CF_BIN)], check=True)
    CF_BIN.chmod(0o755)
if globals().get("CLOUDFLARE_PROCESS") and CLOUDFLARE_PROCESS.poll() is None:
    CLOUDFLARE_PROCESS.terminate()
    CLOUDFLARE_PROCESS.wait(timeout=20)
cf_log = LOG_DIR / "cloudflared.log"
cf_handle = open(cf_log, "w", buffering=1)
CLOUDFLARE_PROCESS = subprocess.Popen([str(CF_BIN), "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{LLAMA_PORT}"], stdout=cf_handle, stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(90):
    time.sleep(2)
    text = cf_log.read_text(errors="ignore") if cf_log.exists() else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
    if match:
        candidate = match.group(0)
        try:
            hr = requests.get(candidate + "/health", headers=AUTH_HEADERS, timeout=10)
            if hr.status_code == 200:
                PUBLIC_URL = candidate
                break
        except requests.RequestException:
            pass
if not PUBLIC_URL:
    print(cf_log.read_text(errors="ignore")[-3000:])
    raise RuntimeError("Tunnel Cloudflare non disponible")
(ROOT / "cloudflare_url.txt").write_text(PUBLIC_URL)
print("API publique:", PUBLIC_URL + "/v1")


In [ ]:
# 8. Heartbeat HMAC, configuration OpenCode sûre et maintien automatique

def heartbeat_endpoint(vps):
    base = vps if "://" in vps else "http://" + vps
    return base.rstrip("/") + "/heartbeat"

def send_heartbeat():
    payload = json.dumps({"url": PUBLIC_URL, "api_key": API_KEY, "model": MODEL_ALIAS, "account": "colab", "timestamp": int(time.time())}, separators=(",", ":")).encode()
    signature = hmac.new(HB_SECRET.encode(), payload, hashlib.sha256).hexdigest()
    response = requests.post(heartbeat_endpoint(VPS), data=payload, headers={"Content-Type": "application/json", "X-Signature": signature}, timeout=20)
    response.raise_for_status()
    print("heartbeat OK")

send_heartbeat()
_stop = globals().get("_stop")
if _stop:
    _stop.set()
_stop = threading.Event()
def heartbeat_loop():
    while not _stop.wait(60):
        try:
            send_heartbeat()
        except Exception as exc:
            print("heartbeat retry:", type(exc).__name__, exc)
HEARTBEAT_THREAD = threading.Thread(target=heartbeat_loop, daemon=True)
HEARTBEAT_THREAD.start()

safe_client_config = {
    "RAVENX_BASE_URL": PUBLIC_URL + "/v1",
    "RAVENX_API_KEY": "<valeur du secret Colab hex>",
    "model": "ravenx/ravenx-chaos-agent-qwen3.8-27b",
    "client": "OpenCode TUI/CLI",
}
print(json.dumps({"status": "READY", "model": MODEL_ALIAS, "gpu": GPU_NAME, "gpu_layers": SELECTED_GPU_LAYERS, "context": CONTEXT, "heartbeat_every_s": 60, "opencode": safe_client_config}, indent=2, ensure_ascii=False))
print("Service maintenu. Interrompre cette cellule pour arrêter le maintien, puis Runtime > Disconnect and delete runtime.")
try:
    while SERVER_THREAD.is_alive() and CLOUDFLARE_PROCESS.poll() is None:
        time.sleep(30)
except KeyboardInterrupt:
    print("Maintien interrompu; les processus restent actifs jusqu'à la fermeture du runtime.")


## Client sélectionné : OpenCode TUI/CLI

OpenCode utilise l’API OpenAI compatible avec les variables `RAVENX_BASE_URL` et `RAVENX_API_KEY`. La configuration complète est fournie dans le dépôt (`opencode.json`) et dans le README. OpenCode peut lire/éditer les fichiers, exécuter le shell et se connecter à des serveurs MCP ; ces opérations sont réalisées côté machine cliente ou VPS, jamais dans le runtime Colab.

La dernière cellule affiche `status: READY`, l’URL publique `/v1`, le modèle et le profil GPU sans révéler le secret. Le service reste actif tant que le runtime Colab et la cellule de maintien restent actifs.
